# Week 3 discussion — gap-tolerance sensitivity (03_02, section 8)


## What changed

The notebook's challenge (section 8) reads "relaxing the gap tolerance from 500 m to
250 m" while also saying to keep the **endpoint connector limit** unchanged. Section 4
of the same notebook (and `AGENTS.md`) actually define two separate thresholds for the
first run: a **50 m** limit for gap connectors between disconnected components of the
same layer, and a separate **500 m** limit for endpoint connectors (start/end to nearest
network point). Since the endpoint limit was already 500 m and is explicitly held fixed,
"the gap tolerance" being relaxed can only be the 50 m component-gap threshold — the
"500 m" in the prompt appears to be a copy-paste of the 500 m endpoint figure from
section 2's preliminary framing. This run therefore relaxes **gap_tolerance_m: 50 → 250**
and keeps `endpoint_connector_limit_m = 500`, endpoints, and the $10/m rate unchanged,
per the rest of the instruction. This assumption is worth flagging to an instructor if
250 m was meant to replace the endpoint limit instead.

Default run: `python week3_routing.py` -> `outputs/default/`
Rerun: `python week3_routing.py --gap-tolerance-m 250 --out outputs/gap_tolerance_250m` -> `outputs/gap_tolerance_250m/`
Both runs and their five deliverables (`routes.gpkg`, `connectors.gpkg`,
`route_comparison.csv`, `route_map.png`, `routing_report.md`) are preserved side by side
under `outputs/`.


## Results comparison

| route    | 50 m gap tolerance (`outputs/default`)      | 250 m gap tolerance (`outputs/gap_tolerance_250m`) | Changed? |
|----------|-----------------------------------------------|------------------------------------------------------|----------|
| baseline | ok, 22,896.8 m, $228,967.56                   | ok, 22,896.8 m, $228,967.56                          | No |
| roads    | ok, 22,897.6 m, 2 connectors, $228,975.93     | ok, 22,897.6 m, 2 connectors, $228,975.93            | No |
| power    | endpoint_too_far (start 1,839 m from network; 9 components -> 3 after repair) | endpoint_too_far (start still 1,839 m from network; 9 components -> **1** after repair) | Internal fragmentation resolved, but status unchanged |
| rail     | ok, 25,902.4 m, 2 connectors (567.5 m), $259,023.67 | ok, 25,902.4 m, 2 connectors (567.5 m), $259,023.67 | No |


## Answers

**1. Which connections came from mapped lines and which were invented by the model?**
Almost the entire length of every successful route is real, digitized OSM
infrastructure. Baseline uses zero connectors (it's the already-continuous shortened
alignment from 03_01). No successful route uses a `gap_connector` — every connector on
a route is an `endpoint_connector` linking a fixed endpoint to the nearest mapped line
(checked in `connectors.gpkg`). Roads uses two tiny endpoint connectors (0.59 m and
0.72 m, 1.31 m total, about 0.006% of its length) — sub-metre snapping of the endpoints
onto the mapped road. Rail uses two much longer endpoint connectors: 194.8 m at the
start and 372.7 m at the end (567.5 m total, about 2.2% of the route), both inside the
500 m endpoint limit. The rail layer's own internal breaks were closed by gap repair
(3 components -> 1), but none of those gap connectors lies on the shortest rail path.
All connector geometry is hypothetical new construction and is drawn distinctly
(`connectors.gpkg`, dashed lime on the map) so it isn't mistaken for verified
infrastructure.

*Correction note (review):* an earlier version of this answer said rail used one gap
connector and one endpoint connector, and described the roads connectors as gap-sized.
The saved `connectors.gpkg` shows all four used connectors are `endpoint_connector`.

**2. Did an unavailable candidate indicate missing data, a real gap, or a restrictive
threshold?**
Power (`endpoint_too_far`) is the unavailable candidate in both runs. Relaxing the gap
tolerance from 50 m to 250 m fully merged the power layer's own fragments (9 raw pieces
→ 1 connected component, versus 3 at 50 m) — so internal fragmentation was indeed partly
a threshold effect. But the route still fails, because the start endpoint sits 1,839 m
from the nearest power line, far beyond both the relaxed 250 m gap tolerance and the
fixed 500 m endpoint limit. With only 15 power features in the whole study rectangle,
that reflects genuinely sparse power-line coverage near this particular endpoint — a
real spatial/data gap, not something a larger *component*-gap threshold can fix. Only
raising the (deliberately fixed) endpoint limit itself would change that outcome, and
doing so hides how far the real infrastructure actually is from the requested corridor.

**3. Why does doubling a common cost per metre double costs without changing their
ranking?**
Cost is `length_m * cost_per_m_usd` with one shared, positive rate applied identically to
every candidate. Multiplying every route's length by the same constant is a uniform
linear scaling — it preserves the order of the values (if length A < length B, then
`k*A < k*B` for any k > 0). Doubling the rate to $20/m would double every row's cost but
leave baseline still the cheapest, roads a close second, and rail the most expensive, in
the same order as before. Ranking could only change under this model if the *rate*
differed by candidate (e.g., a rail-specific easement premium or a roads-specific
permitting discount), which this simplified exercise deliberately excludes.

**4. Which route would you investigate further, and what evidence would you need?**
Roads is the strongest candidate for follow-up: it is essentially tied with the raw
baseline in length (22,897.6 m vs 22,896.8 m, <1 m apart) while running along existing
road corridor rather than a purely theoretical straight alignment, and it needed only
~1.3 m of invented connector. Before treating it as buildable I'd want (a) road
right-of-way/permit confirmation from the relevant authority, since proximity to a road
implies no legal access; (b) a check of the two connector locations and any road
crossings against current satellite imagery for bridges, culverts, or grade separations
that could invalidate the model's flat 2D-crossing assumption; and (c) a higher-resolution
resurvey of OSM geometry near the endpoints, since the connectors are sub-meter and could
shrink to zero with better-aligned source data. Rail is the second-best available
alternative for corridor diversity and is worth checking specifically for the ~567 m of
new endpoint-connector construction it requires (194.8 m at the start, 372.7 m at the
end: is the rail line really that far from the endpoints, or is a spur unmapped?) and for railway easement/access rules. Power is not worth pursuing
along this alignment without new data — its start point is nearly 2 km from the nearest
mapped line, which is a genuine coverage gap this exercise's rules are correct not to
paper over.
